In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from tqdm import tqdm
tqdm.pandas()

from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

from lightgbm import LGBMClassifier
from xgboost import  XGBClassifier
from sklearn.ensemble import  RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import f1_score


import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None  # default='warn'
#pd.set_option('display.float_format', lambda x: '%.3f' % x)
plt.rcParams["figure.figsize"] = (12, 8)
pd.set_option('display.max_columns', None)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def check_target_distribution(df, target_column):
  df[target_column].hist(bins=35, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
df = df.drop('Order_ID', axis=1)

In [ ]:
df.info()

In [ ]:
def check_missing_values(df):
    """
    Detailed report of missing values:
    Shows the count and percentage for each column that has missing data.
    """
    # Calculate missing counts and percentages
    missing_count = df.isnull().sum()
    missing_percent = 100 * df.isnull().sum() / len(df)

    # Create a summary table
    missing_table = pd.concat([missing_count, missing_percent], axis=1)
    missing_table.columns = ['Missing Count', 'Percentage (%)']

    # Filter to show only columns with missing values
    missing_table = missing_table[missing_table['Missing Count'] > 0].sort_values(by='Percentage (%)', ascending=False)

    if not missing_table.empty:
        print("Missing Values Summary:")
        print(missing_table)


    else:
        print("No Missing Values Found.")

check_missing_values(df)

In [ ]:
# Task 2: Write your code here:
# so     Delivery_Time  Weather            Traffic_Level              Time_of_Day                Courier_Experience_yrs


In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer

# 1. Identify columns by their data types
# Numerical columns (e.g., height, weight, stats)
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

# Categorical columns (e.g., types, colors, categories)
cat_cols = df.select_dtypes(include=['object', 'category']).columns

# 2. Initialize Imputer for numerical data using the median
num_imputer = SimpleImputer(strategy='median')

# 3. Initialize Imputer for categorical data using a constant value
# This treats 'None' as a distinct category instead of a missing value
cat_imputer = SimpleImputer(strategy='constant', fill_value='none')

# 4. Apply transformation to the dataframe
df[num_cols] = num_imputer.fit_transform(df[num_cols])
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

# Verify that no missing values remain
print(df.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:

# Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
df

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder

categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = OneHotEncoder()
    df[col] = le.fit_transform(df[col])
df

In [ ]:
# 3. Do we have categorical columns?
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
# Task 5: Write your code here:

from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 6: Write your code here:
#Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)

# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
    print("Target Distribution:")
    print(df[target_column].value_counts(normalize=True))
    sns.histplot(x=df[target_column])
    plt.title("Target Distribution")
    plt.show()

check_target_imbalance(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time",axis=1)
y = df['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:

In [ ]:
# Calculate the majority class baseline
majority_class = y.value_counts().idxmax()
baseline_pred = [majority_class] * len(y)

# Evaluate the baseline
baseline_f1 = f1_score(y, baseline_pred, average='weighted')
print(f"Baseline F1-Score (majority class): {baseline_f1:.4f}")


In [ ]:
# Define classification model
model =RandomForestClassifier(n_estimators=100)



    # Stratified 5-Fold Cross-Validation
skf = KFold(n_splits=5)
      for train_index, test_index in skf.split(X, y):
        # Split data into training and testing sets
        X_Train, X_Test = X.loc[train_index, :], X.loc[test_index, :]
        y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]
        # Train the model
        model.fit(X_Train, y_Train)
        # Predict on the test set
        y_pred = model.predict(X_Test)


In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here:

In [ ]:
# Define classification models
models = {
    "Logistic Regression": LogisticRegression(max_iter=10000),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=100),
    "CatBoost Classifier": CatBoostClassifier(verbose=0)
}

models["Ensemble"] = VotingClassifier(estimators=[
        ('lr', models["CatBoost Classifier"]), ('rf', models["Logistic Regression"]), ('gnb', models["Random Forest Classifier"])], voting="soft")

for model_name, model in models.items():
    scores_accuracy = []
    scores_precision = []
    scores_recall = []
    scores_f1 = []

    # Stratified 5-Fold Cross-Validation
    skf = KFold(n_splits=5)
    for train_index, test_index in skf.split(X, y):
        # Split data into training and testing sets
        X_Train, X_Test = X.loc[train_index, :], X.loc[test_index, :]
        y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]
        # Train the model
        model.fit(X_Train, y_Train)
        # Predict on the test set
        y_pred = model.predict(X_Test)

        # Calculate metrics
        scores_f1.append(f1_score(y_Test, y_pred, average='weighted'))

    # Print the results
    print(f"{model_name} F1-Score: {np.mean(scores_f1):.4f}")
    print("\n")

